# CoT-ENEM — Fase 4 no Colab

Executa Complicate e/ou Diversify como ramos independentes sobre o CoT inicial congelado da Fase 3. Cada ramo mantém candidatos, votos e resultados próprios. Para economizar cota, execute uma estratégia por sessão.

In [ ]:
# Edite somente estes valores.
REPOSITORY_URL = "https://github.com/cidadaofred/cot-enem.git"
BRANCH = "master"
DRIVE_ROOT = "/content/drive/MyDrive/cot-enem"
STRATEGIES = ["complicate"]  # use ["diversify"] ou ambas
LIMIT = 1  # smoke test; use None somente após validar


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os, shutil, subprocess
PROJECT = Path("/content/cot-enem").resolve()
if PROJECT != Path("/content/cot-enem"):
    raise RuntimeError(f"Caminho inseguro: {PROJECT}")
os.chdir("/content")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(PROJECT)],
    text=True, capture_output=True,
)
if clone.returncode != 0:
    raise RuntimeError(f"Falha no git clone:\n{clone.stderr.strip()}")
os.chdir(PROJECT)
os.environ["HF_HOME"] = "/content/hf-cache"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["python", "-m", "cot_enem.cli", "verify", "--config", "configs/colab.yaml"], check=True)


In [ ]:
ROOT = Path(DRIVE_ROOT)
PHASE3_CANDIDATES = ROOT / "outputs/datasets/phase3_candidates.jsonl"
SHARED_SEEDS = ROOT / "outputs/datasets/shared/initial_cot_seeds.jsonl"
if not PHASE3_CANDIDATES.exists():
    raise FileNotFoundError(f"Candidatos da Fase 3 ausentes: {PHASE3_CANDIDATES}")
for strategy in STRATEGIES:
    if strategy not in {"complicate", "diversify"}:
        raise ValueError(f"Estratégia inválida: {strategy}")
print(f"Entrada reutilizada: {PHASE3_CANDIDATES}")


In [ ]:
for strategy in STRATEGIES:
    branch = ROOT / "outputs/datasets" / strategy
    candidates = branch / "candidates.jsonl"
    votes = branch / "judge_votes.jsonl"
    output = branch / "results.jsonl"
    command = [
        "python", "-m", "cot_enem.cli", "ensemble-evolve",
        "--config", "configs/colab.yaml",
        "--strategy", strategy,
        "--initial-candidates", str(PHASE3_CANDIDATES),
        "--seeds", str(SHARED_SEEDS),
        "--candidates", str(candidates),
        "--votes", str(votes),
        "--output", str(output),
    ]
    if LIMIT is not None:
        command += ["--limit", str(LIMIT)]
    print(f"\n=== Executando {strategy} ===")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"{strategy} terminou com código {returncode}")


## Auditoria em CPU

Valida cada ramo sem carregar nenhum modelo.

In [ ]:
import sys, yaml
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))
from cot_enem.audit import audit_phase4_branch
from cot_enem.dataset.schema import Strategy
with Path("configs/colab.yaml").open("r", encoding="utf-8") as file:
    judge_models = yaml.safe_load(file)["model"]["judge_names"]
for strategy_name in STRATEGIES:
    branch = ROOT / "outputs/datasets" / strategy_name
    summary = audit_phase4_branch(
        SHARED_SEEDS,
        branch / "candidates.jsonl",
        branch / "judge_votes.jsonl",
        branch / "results.jsonl",
        judge_models,
        Strategy(strategy_name),
        require_complete=LIMIT is None,
    )
    print(f"\nAUDITORIA {strategy_name.upper()}: APROVADA")
    for key, value in summary.items():
        print(f"{key:24}: {value}")
